In [2]:
import pandas as pd
import numpy as np 
import os
import geopandas as gpd
import math

In [4]:
path = "../../data/data_biaised/"
# dir_list = os.listdir(path)
# data_np = {}
# data = {}
data_all = {}

# for x in os.listdir(path):
#     if x.endswith(".csv"):
#         if x.endswith("np.csv"):
#         # Prints only text file present in My Folder
#             data_np[x] = pd.read_csv(path+x, sep=";")
#         else : 
#             data[x] = pd.read_csv(path+x, sep=";")
#         # data_all[x] = pd.read_csv(path+x, sep=";")
        
        
arr = os.listdir(path)   
data_all = {}

for file in arr :
    if file.endswith('.csv'):
        if file.endswith('np.csv'):
            tag = file.split('_')[2]+'_np'
        else :
            tag = file.split('_')[2][:-4]

        df = pd.read_csv(os.path.join(path,file),sep=";",dtype=str)
        df['tag'] = tag
        data_all[f'{tag}'] = df


In [5]:
df_revenus = pd.read_csv("../../data/insee_revenu/BASE_TD_FILO_DISP_IRIS_2020.csv",sep=";",dtype=str)[["IRIS","DISP_MED20"]]
df_iris = gpd.read_file('../../data/zones_geographiques/iris/CONTOURS-IRIS.shp')


In [6]:
df_revenus[df_revenus["DISP_MED20"]=="ns"] = np.nan
df_revenus[df_revenus["DISP_MED20"]=="nd"] = np.nan

df_revenus["DISP_MED20"] = df_revenus["DISP_MED20"].astype(float)

df_revenus = df_revenus.rename({"IRIS":"CODE_IRIS"},axis=1).astype(str)
df_revenus = df_revenus.dropna(subset='CODE_IRIS')
df_revenus = df_revenus.drop_duplicates(subset='CODE_IRIS')

In [7]:
## Distance entre deux point d'une sphère => coordonnées exprimées en WGS84 (degrès décimaux): comprend un modèle de la terre  d'où le calcul de distance angulaire 
def calculer_distance_haversine(lat1, lon1, lat2, lon2):

    try : 
        # Rayon de la Terre en kilomètres
        R = 6371.0

        # Conversion des degrés en radians
        dLat = math.radians(lat2 - lat1)
        dLon = math.radians(lon2 - lon1)
        rLat1 = math.radians(lat1)
        rLat2 = math.radians(lat2)

        # Formule de Haversine
        a = math.sin(dLat / 2)**2 + math.cos(rLat1) * math.cos(rLat2) * math.sin(dLon / 2)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

        # Distance totale
        distance = R * c
        return distance
    except Exception as e : 
        print(f"Erreur lors du calcul de la distance : {e}")
        return np.nan
    

In [ ]:
# df_test = data_all[x]

# df_test = df_test.rename({'CODE_IRIS':'CODE_IRIS_geocoded'},axis=1).astype(str)

# gdf = (gpd.GeoDataFrame(df_test, geometry=gpd.points_from_xy(df_test.longitude, df_test.latitude)).set_crs(epsg=4326)).to_crs(epsg=2154)
# df_join_iris = gpd.sjoin(gdf, df_iris[['CODE_IRIS','geometry']], how="left", op='within')
# df_join_iris.drop('index_right', axis=1, inplace=True)
# df_test = df_join_iris.rename({'CODE_IRIS':'CODE_IRIS_init'},axis=1)
# print(len(df_test))
# ## ----------------------------------
# ## Ajout DISP_MED20 pour le code iris geocodé et le code iris initial 

# # Jointure pour récupérer le revenu médian associé à l'iris initial : 
# df_test = df_test.dropna(subset="CODE_IRIS_geocoded")
# print(len(df_test))
# df_rev_geoc = pd.merge(df_test,df_revenus, how="left",left_on="CODE_IRIS_geocoded",right_on="CODE_IRIS")
# # df_rev_geoc = df_test.merge(df_revenus,how="left",left_on="CODE_IRIS_geocoded",right_on="CODE_IRIS")
# print(len(df_rev_geoc))
# df_rev_geoc = df_rev_geoc.rename({'DISP_MED20':'DISP_MED20_geocoded'},axis=1)
# df_rev_geoc = df_rev_geoc.drop('CODE_IRIS',axis=1)
# print(len(df_rev_geoc))

In [8]:
## X,Y de référence : 'latitude','longitude'
## X,Y à évaluer : 'x','y'
data_w_info = {}
data = data_all
for f in data_all: 
    df= data_all[f]
    tag = df.loc[0,'tag']
    # tag = f.split('_')[2].split('.')[0]
    # df['tag'] = tag 

    ##-----------------------------------
    ## Ajout CODE IRIS initial 

    df = df.rename({'CODE_IRIS':'CODE_IRIS_geocoded'},axis=1).astype(str)

    gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude)).set_crs(epsg=4326)).to_crs(epsg=2154)
    df_join_iris = gpd.sjoin(gdf, df_iris[['CODE_IRIS','geometry']], how="left", op='within')
    df_join_iris.drop('index_right', axis=1, inplace=True)
    df = df_join_iris.rename({'CODE_IRIS':'CODE_IRIS_init'},axis=1)
    ## ----------------------------------
    ## Ajout DISP_MED20 pour le code iris geocodé et le code iris initial 
    
    # Jointure pour récupérer le revenu médian associé à l'iris initial : 
    df = df.dropna(subset="CODE_IRIS_geocoded")
    df_rev_geoc = df.merge(df_revenus,how="left",left_on="CODE_IRIS_geocoded",right_on="CODE_IRIS")
    df_rev_geoc = df_rev_geoc.rename({'DISP_MED20':'DISP_MED20_geocoded'},axis=1)
    df_rev_geoc = df_rev_geoc.drop('CODE_IRIS',axis=1)
    
    # Jointure pour récupérer le revenu médian associé à l'iris géocodé : 
    df_rev_init = df_rev_geoc.dropna(subset="CODE_IRIS_init")
    df_rev_init = df_rev_init.merge(df_revenus,how="left",left_on="CODE_IRIS_init",right_on="CODE_IRIS")
    df_rev_init = df_rev_init.rename({'DISP_MED20':'DISP_MED20_init'},axis=1)

    df_rev = df_rev_init.copy()

    df_rev["DISP_MED20_geocoded"] = df_rev["DISP_MED20_geocoded"].astype(float)
    df_rev["DISP_MED20_init"] = df_rev["DISP_MED20_init"].astype(float)
    

    ## ----------------------------------
    ## Calcul distance et diff revenu 

    df_rev["y"] = df_rev["y"].astype(float)
    df_rev["x"] = df_rev["x"].astype(float)

    df_rev["latitude"] = df_rev["latitude"].astype(float)
    df_rev["longitude"] = df_rev["longitude"].astype(float)
    liste_ini =[]
    liste_geoc = [] 
    for i in df_rev.index : 

        lon1 = df_rev.at[i,"x"]
        lat1 = df_rev.at[i,"y"]
        lon2 = df_rev.at[i,"longitude"]
        lat2 = df_rev.at[i,"latitude"]

        # if pd.isna(lon1) or pd.isna(lat1): 
        #     liste_geoc.append(i)
        # if pd.isna(lon2) or pd.isna(lat2) :
        #     liste_ini.append(i)

        df_rev.at[i,"distance_km"] = calculer_distance_haversine(lat1, lon1, lat2, lon2)
        df_rev.at[i,"distance_m"] = df_rev.loc[i,'distance_km']*1000

        ## ----------------------------------
        df_rev.at[i,"diff_revenu"] =pd.to_numeric(df_rev.loc[i,"DISP_MED20_init"]) - pd.to_numeric(df_rev.loc[i,"DISP_MED20_geocoded"])

    data_w_info[tag] = df_rev

# file = 'df_ref_RETRAITE.csv'
# file.split('_')[2].split('.')[0]

/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3361: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if (await self.run_code(code, result,  async_=asy)):
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3361: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if (await self.run_code(code, result,  async_=asy)):
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3361: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if (await self.run_code(code, result,  async_=asy)):
/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3361: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` pa

In [9]:
df_all = pd.concat(data_w_info,axis=0).reset_index()
len(df_all)

19000

In [10]:

df_all['distance_m'] = df_all['distance_m'].astype(float)
df_all['diff_revenu'] = df_all['diff_revenu'].astype(float)
df_mean = df_all.groupby('tag')[['distance_km','diff_revenu']].mean().apply(lambda x : np.round(x,3)).rename({'distance_km':'dist_moy_km','diff_revenu':'diff_moy_revenu'},axis=1)
df_med = df_all.groupby('tag')[['distance_km','diff_revenu']].median().apply(lambda x : np.round(x,3)).rename({'distance_km':'dist_med_km','diff_revenu':'diff_med_revenu'},axis=1)

In [11]:
all_tag = []
for tag in data_w_info: 
    all_tag.append(tag)

prc_chgmt_iris = {}
for tag in all_tag: 
    nb_changement = 0
    for i in df_all[df_all['tag']==tag].index:
        if df_all.loc[i,'CODE_IRIS_init'] != df_all.loc[i,'CODE_IRIS_geocoded']:
            nb_changement+=1
    prc_chgmt_iris[tag] = (nb_changement/len(df_all[df_all['tag']==tag]))*100

df_shift = pd.DataFrame.from_dict(prc_chgmt_iris, orient="index")
df_shift = df_shift.rename(columns={ 
    df_shift.columns[0]: " % de changement d'iris "
})


In [12]:
df_shift[" % de changement d'iris "] = df_shift[" % de changement d'iris "].apply(lambda x : np.round(x,3))

df_final = pd.concat([df_mean,df_med,df_shift],axis=1)
df_final = df_final.reindex(['dist_moy_km','dist_med_km', 'diff_moy_revenu',  'diff_med_revenu'," % de changement d'iris "],axis=1)

In [13]:
df_final_mean = pd.concat([df_mean,df_shift],axis=1)

In [14]:
df_final_mean

,dist_moy_km,diff_moy_revenu,% de changement d'iris
APPT,10.682,-16.064,7.0
BAT,1.532,-128.027,9.4
CENTRE,3.308,-32.562,6.2
CHEZ,2.406,-138.953,8.1
CHEZ_np,8.753,-113.764,11.9
HOPITAL,2.238,-50.775,6.6
HOTEL,2.810,-55.988,7.2
HOTEL_np,6.169,-74.771,9.1
MAISON,4.046,-54.663,7.6
MME,3.238,-20.930,5.5


In [ ]:
test = df_all[df_all['distance_km']>5]
# test = df_all.groupby('numero_uai')['distance_m']
# test[test['distance_m']>] 
# df_count = pd.DataFrame(test.groupby('numero_uai').size())
# print(len(df_count))
# df_count = df_count.rename({'0':'count'},axis=1).reset_index()
# df_count.columns
# res = df_count.groupby(0).size()
# res 
# df_mean = df_all.groupby('tag')[['distance_m','diff_revenu']].mean()



# df_count = pd.DataFrame(test.groupby('tag').size())
# print(len(df_count))
# df_count = df_count.rename({'0':'count'},axis=1).reset_index()
# # res = df_count.groupby(0).size()
# # res 
# df_count

## Visu

In [ ]:
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

long_df = pd.melt(df_all, value_vars=['DISP_MED20_init', 'DISP_MED20_geocoded'], var_name='Type', value_name='Revenu')

# fig = px.density_contour(long_df, x='Revenu', y='Type', marginal_x='histogram', color='Type',
#                          title='Comparaison des distributions de revenu: INIT vs GEOCODED')

# fig.show()

# positions = pd.melt(df, id_vars=['adresse'], value_vars=['Numbers','Street type','Zipcode','City','Add. info'], 
#                     var_name='Type', value_name='Position_relative')


# Graphique de densité corrigé
sns.kdeplot(data=long_df, x='Revenu', hue='Type', fill=True, common_norm=False, alpha=0.5)
plt.title('Density comparison of median income for our reference')#Densité de la position relative des éléments de l\'adresse')
plt.xlabel('Revenu')
plt.ylabel('Density')
plt.show()


In [ ]:
fig = px.scatter(df_all[df_all['distance_km']<10],x='distance_km',y='diff_revenu')
fig.show()

## Impact nettoyage sur distance 

In [15]:
# df_all.columns

# df_all = df_all.drop(columns = ['level_0', 'level_1', 'Unnamed: 0', 'index'])

df_all_toclean = df_all.copy()[["numero_uai",'adresse','code_postal_uai','libelle_commune','requete','longitude','latitude']]
df_all_toclean = df_all_toclean.rename(columns = {'longitude': 'longitude_ref',
                                                  'latitude' : 'latitude_ref'})

In [18]:
%run cleaning_functions.py

#### Application de l'algo de cleaning

In [31]:
df = df_all_toclean.dropna()

voirie = ["RUE","COURS","COUR","VOIE","RUELLE","ESPLANADE",
          "PLACE","SQUARE","SQ","ROND POINT","PL",
          "IMPASSE", "ALLEE", "CHEMIN" ,"ROUTE","RTE","IMP","PROMENADE","ALLE","ALL",
          "AVENUE", "BOULEVARD","BD","AVE","BLD","BVD","BLV","AVN","BV",
          "FERME","DOMAINE","LIEU DIT","QUARTIER","QUR"]

numeros = ["1","2","3","4","5","6","7","8","9","0"]

bruit = [ "RESIDENCE","CHEZ","HOPITAL","SDF","RES","MME","BAT","MAISON","MR","HOTEL",
         "LOTISSEMENT","CENTRE","QUARTIER","APPT","APT","SANTE", "RETRAITE", "HOP","TRANSFERT"]

df['adresse'] = df['adresse'].astype(str).apply(lambda x: x.upper())
df['requete'] = df['requete'].astype(str).apply(lambda x: x.upper())
df = prep_adresse(df)

df['voirie'] = df.apply(lambda x: find_attribute(voirie, x['adresse']), axis=1)
df['numeros'] = df.apply(lambda x: find_attribute(numeros, x['adresse'],is_number=True), axis=1)
df['bruit'] = df.apply(lambda x: find_attribute(bruit, x['adresse']), axis=1)



In [ ]:
# %run cleaning_functions.py
df = filtre_num_voirie(df)

In [33]:
df['pos_voirie'] = df.apply(lambda x: find_pos_attributes(x['voirie'], x['adresse']), axis=1)
df['pos_numeros'] = df.apply(lambda x: find_pos_attributes(x['numeros'], x['adresse'],is_number=True), axis=1)
df['pos_bruit'] = df.apply(lambda x: find_pos_attributes(x['bruit'], x['adresse'],is_bruit=True), axis=1)

In [34]:
df = filtre_bruit_to_voirie(df) 
df = clean_wrong_voirie(df)


df['pos_prc_voirie'] = df.apply(lambda x: find_pos_prc_attributes(x['voirie'], x['adresse']), axis=1)
df['pos_prc_numeros'] = df.apply(lambda x: find_pos_prc_attributes(x['numeros'], x['adresse'],is_number=True), axis=1)
df['pos_prc_bruit'] = df.apply(lambda x: find_pos_prc_attributes(x['bruit'], x['adresse']), axis=1) 

df_final_cleaned = filtre_elem_adresse(df)

In [35]:
df_comparatif = df_all[df_all['distance_km']>1][['numero_uai','tag']]
id_grosse_distance = list(df_comparatif['numero_uai'])#df_comparatif['numero_uai'].tolist()
# df_all.columns

In [36]:
df_final_filtered_mul = df_final_cleaned[df_final_cleaned['numero_uai'].isin(id_grosse_distance)]
# df_final_filtered_uni = df_final_filtered_mul.drop_duplicates(subset="numero_uai")
# len(df_final_filtered)#.drop_duplicates(subset="numero_uai"))
print(len(df_final_filtered_mul))

1016


In [37]:
df_final_filtered = df_final_filtered_mul.copy()
df_final_filtered['adresse_cleaned'] = df_final_filtered['numeros'] + ' ' + df_final_filtered['voirie'] + ' ' + df_final_filtered['elem_adresse']

df_final_filtered = df_final_filtered.rename({ 'numero_uai' :  'pseudo_provisoire', 
                            'code_postal_uai': 'codepost',
                           'libelle_commune': 'nom_commune_postal'}, axis=1) 

df_cleaned_to_geoloc = df_final_filtered[['pseudo_provisoire','adresse_cleaned','codepost','nom_commune_postal','longitude_ref','latitude_ref']]

df_cleaned_to_geoloc['requete'] = df_cleaned_to_geoloc['adresse_cleaned'] + ' ' + df_cleaned_to_geoloc['codepost'].astype(str) + ' ' + df_cleaned_to_geoloc['nom_commune_postal']

df_cleaned_to_geoloc['nom_commune_postal'] = df_cleaned_to_geoloc['nom_commune_postal'].astype(str).apply(lambda x: x.upper())
df_cleaned_to_geoloc['requete'] = df_cleaned_to_geoloc['requete'].astype(str).apply(lambda x: x.upper())

/tmp/ipykernel_3586/3096813217.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned_to_geoloc['requete'] = df_cleaned_to_geoloc['adresse_cleaned'] + ' ' + df_cleaned_to_geoloc['codepost'].astype(str) + ' ' + df_cleaned_to_geoloc['nom_commune_postal']
/tmp/ipykernel_3586/3096813217.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned_to_geoloc['nom_commune_postal'] = df_cleaned_to_geoloc['nom_commune_postal'].astype(str).apply(lambda x: x.upper())
/tmp/ipykernel_3586/3096813217.py:13: S

In [50]:
# row_to_drop = []
# for i in df_cleaned_to_geoloc.index : 
#     if df_cleaned_to_geoloc.loc[i,'adresse_cleaned'] == "":
#         row_to_drop.append(i)
#     if ' nan ' in df_cleaned_to_geoloc.loc[i,'adresse_cleaned'] : 
#         row_to_drop.append(i)


In [38]:
import requests 
%run -i geocoding_functions.py

In [39]:
df_geocoded_cleaned = geocode(df_cleaned_to_geoloc)
df_geocoded_cleaned.to_csv('./from_hegp/biais/ref_biais_cleaned_dist1km.csv',sep=";")

Proceed to geocode on address...


/home/jovyan/work/00_canc_air/canc_air/geocodeur/geocoding_functions.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.at[i, 'x'] = str(response["features"][0]["geometry"]["coordinates"][0])
/home/jovyan/work/00_canc_air/canc_air/geocodeur/geocoding_functions.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.at[i, 'y'] = str(response["features"][0]["geometry"]["coordinates"][1])
/home/jovyan/work/00_canc_air/canc_air/geocodeur/geocoding_functions.py:16: SettingWithCopyWarning: 
A value is trying

Proceed geocoding at the 5100th row
Proceed geocoding at the 5500th row
Proceed geocoding at the 5900th row
Proceed geocoding at the 6200th row
Proceed geocoding at the 6300th row
Proceed geocoding at the 10100th row
Proceed geocoding at the 10500th row
Proceed geocoding at the 10900th row
Proceed geocoding at the 13200th row
Proceed geocoding at the 13300th row
Proceed geocoding at the 14600th row
Proceed geocoding at the 17200th row
Proceed geocoding at the 17700th row


In [42]:
df_geocoded_cleaned

,pseudo_provisoire,adresse_cleaned,codepost,nom_commune_postal,longitude_ref,latitude_ref,requete,x,y,score,trust_score,street,city,pc_city,code_dept,address
27,0420817K,CHEMIN DES TOURETTES,42100,SAINT-ETIENNE,4.264489,45.444173,CHEMIN DES TOURETTES 42100 SAINT-ETIENNE,4.402664,45.405116,0.5567954545454545,middle,CHEMIN DE LAYA,SAINT-ÉTIENNE,42100,42,CHEMIN DE LAYA 42100 SAINT-ÉTIENNE
144,0861184V,6 AVENUE DE PROVENCE,86500,MONTMORILLON,0.852451,46.418054,6 AVENUE DE PROVENCE 86500 MONTMORILLON,0.852451,46.418054,0.9535645454545454,high,6 AVENUE DE PROVENCE,MONTMORILLON,86500,86,6 AVENUE DE PROVENCE 86500 MONTMORILLON
238,0572829R,RUE DES GRANDS BOIS,57700,HAYANGE,6.065559,49.316934,RUE DES GRANDS BOIS 57700 HAYANGE,6.067428,49.316888,0.9621481818181817,high,RUE DES GRANDS BOIS,HAYANGE,57700,57,RUE DES GRANDS BOIS 57700 HAYANGE
262,0580618G,PLACE ELSA TRIOLET,58160,IMPHY,3.265690,46.930411,PLACE ELSA TRIOLET 58160 IMPHY,2.549019,48.943942,0.5352647335423196,middle,PLACE ELSA TRIOLET,SEVRAN,93270,93,PLACE ELSA TRIOLET 93270 SEVRAN
281,0442902R,3 RUE VASCO DE GAMA,44800,SAINT-HERBLAIN,-1.624287,47.221048,3 RUE VASCO DE GAMA 44800 SAINT-HERBLAIN,-1.659157,47.208406,0.6040571074380164,middle,RUE DE LA GARE,SAINT-HERBLAIN,44800,44,RUE DE LA GARE 44800 SAINT-HERBLAIN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18925,0332606D,RUE ERIK SATIE,33270,FLOIRAC,-0.507412,44.842450,RUE ERIK SATIE 33270 FLOIRAC,-0.507724,44.842136,0.9559536363636364,high,RUE ERIK SATIE,FLOIRAC,33270,33,RUE ERIK SATIE 33270 FLOIRAC
18942,0210368L,RUE DE LA MOTTE,21200,BEAUNE,4.870660,47.037919,RUE DE LA MOTTE 21200 BEAUNE,4.871573,47.037795,0.9667381818181817,high,RUE DE LA MOTTE,BEAUNE,21200,21,RUE DE LA MOTTE 21200 BEAUNE
18951,0241002J,,24660,COULOUNIEIX-CHAMIERS,0.688870,45.157474,24660 COULOUNIEIX-CHAMIERS,0.688382,45.17294,0.9445436363636364,high,COULOUNIEIX-CHAMIERS,COULOUNIEIX-CHAMIERS,24660,24,COULOUNIEIX-CHAMIERS
18986,0160894K,19 RUE DU 19 MARS 1962,16800,SOYAUX,0.205419,45.654804,19 RUE DU 19 MARS 1962 16800 SOYAUX,0.186294,45.640331,0.49392984478935703,middle,RUE DU 8 MAI 1945,SOYAUX,16800,16,RUE DU 8 MAI 1945 16800 SOYAUX


In [60]:
df_geocoded_cleaned.columns

Index(['pseudo_provisoire', 'adresse_cleaned', 'codepost',
       'nom_commune_postal', 'longitude_ref', 'latitude_ref', 'requete', 'x',
       'y', 'score', 'trust_score', 'street', 'city', 'pc_city', 'code_dept',
       'address', 'distance_cleaned_km'],
      dtype='object')

In [61]:
df_geocoded_cleaned["latitude_ref"] = df_geocoded_cleaned["latitude_ref"].astype(float)
df_geocoded_cleaned["longitude_ref"] = df_geocoded_cleaned["longitude_ref"].astype(float)

df_geocoded_cleaned["y"] = df_geocoded_cleaned["y"].astype(float)
df_geocoded_cleaned["x"] = df_geocoded_cleaned["x"].astype(float)

for i in df_geocoded_cleaned.index : 

    liste_ini =[]
    liste_geoc = [] 

    lon1 = df_geocoded_cleaned.at[i,"x"]
    lat1 = df_geocoded_cleaned.at[i,"y"]
    lon2 = df_geocoded_cleaned.at[i,"longitude_ref"]
    lat2 = df_geocoded_cleaned.at[i,"latitude_ref"]


    df_geocoded_cleaned.at[i,"distance_km"] = calculer_distance_haversine(lat1, lon1, lat2, lon2)
    # df_geocoded_cleaned.at[i,"distance_m"] = df_geocoded_cleaned.loc[i,'distance_km']*1000

In [64]:
df_geocoded_cleaned[['adresse_cleaned','distance_km']]

print(df_all[df_all['distance_notcleaned_km']>1]['distance_notcleaned_km'].median())
print(df_geocoded_cleaned[df_geocoded_cleaned['distance_km']>1]['distance_km'].median())

2.869777304192267
3.1185316862618686


In [53]:
df_all = df_all.rename(columns={'distance_km': 'distance_notcleaned_km'}) 
df_geocoded_cleaned = df_geocoded_cleaned.rename(columns={'distance_km': 'distance_cleaned_km'}) 

In [56]:
df_comparatif = pd.merge(df_geocoded_cleaned, df_all, how ="left",left_on="pseudo_provisoire",right_on="numero_uai")
print(len(df_geocoded_cleaned),len(df_comparatif))

1016 2488


In [58]:
len(df_all.drop_duplicates(subset="numero_uai"))

10757

In [59]:
len(df_all)

19000